# Server Flask su Colab per Unity

Apri questo notebook in VS Code, poi scegli **Select Kernel -> Colab -> New Colab Server**. Le celle gireranno sulla GPU di Colab, non sul Mac.

Importante: il runtime Colab non vede automaticamente i file locali del Mac. Il flusso consigliato e': lavori in VS Code, fai push su GitHub, poi questo notebook fa clone/pull dentro Colab.

In [ ]:
# 1) Configura qui il tuo repository GitHub
# Esempio: REPO_URL = "https://github.com/tuo-utente/ProgettoCG.git"
REPO_URL = ""
PROJECT_DIR = "/content/ProgettoCG"

if not REPO_URL:
    raise ValueError("Metti l'URL del tuo repo GitHub in REPO_URL, poi riesegui questa cella.")

In [ ]:
# 2) Clona o aggiorna il progetto nel filesystem remoto di Colab
import os
from pathlib import Path

if Path(PROJECT_DIR).exists():
    %cd {PROJECT_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} ProgettoCG
    %cd {PROJECT_DIR}

print("Project dir:", os.getcwd())

In [ ]:
# 3) Installa ComfyUI e dipendenze Colab
from pathlib import Path

%cd {PROJECT_DIR}
if not Path("ComfyUI").exists():
    !git clone https://github.com/comfyanonymous/ComfyUI

!pip install -q -r ComfyUI/requirements.txt
!pip install -q -r requirements-colab.txt

print("Dipendenze installate")

In [ ]:
# 4) Scarica i modelli su Colab. Non caricarli dal Mac.
%cd {PROJECT_DIR}
!mkdir -p ComfyUI/models/diffusion_models ComfyUI/models/clip ComfyUI/models/vae

!wget -c -O ComfyUI/models/diffusion_models/z-image-turbo-fp8-e4m3fn.safetensors \
  https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors

!wget -c -O ComfyUI/models/clip/qwen_3_4b.safetensors \
  https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors

!wget -c -O ComfyUI/models/vae/ae.safetensors \
  https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors

!ls -lh ComfyUI/models/diffusion_models ComfyUI/models/clip ComfyUI/models/vae

In [ ]:
# 5) Avvia Flask in background
import os
import subprocess
import time

%cd {PROJECT_DIR}
env = os.environ.copy()
env["COMFYUI_PATH"] = f"{PROJECT_DIR}/ComfyUI"

try:
    FLASK_PROC.terminate()
except NameError:
    pass

log = open("flask.log", "w")
FLASK_PROC = subprocess.Popen(["python", "app.py"], cwd=PROJECT_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
time.sleep(3)
print("Flask PID:", FLASK_PROC.pid)
!tail -n 40 flask.log

In [ ]:
# 6) Apri un tunnel pubblico per Unity con cloudflared
import re
import subprocess
import time
from pathlib import Path

%cd {PROJECT_DIR}
if not Path("cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

try:
    CLOUDFLARED_PROC.terminate()
except NameError:
    pass

CLOUDFLARED_PROC = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = CLOUDFLARED_PROC.stdout.readline()
    if line:
        print(line, end="")
        match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

if not public_url:
    raise RuntimeError("URL cloudflared non trovato. Riesegui la cella o controlla l'output.")

print("\nBASE URL PER UNITY:", public_url)
print("Health:", public_url + "/health")

In [ ]:
# 7) Test rapido dell'API dal notebook
import requests

r = requests.get(public_url + "/health", timeout=20)
print(r.status_code, r.text)

# Test generazione. Puo' richiedere tempo e VRAM.
# payload = {
#     "session_id": "unity-test",
#     "prompt": "a futuristic white sneaker, product render, clean background",
#     "width": 1024,
#     "height": 1024,
#     "steps": 9,
#     "cfg": 1.0,
# }
# r = requests.post(public_url + "/generate-image", json=payload, timeout=600)
# print(r.status_code)
# print(r.text)